In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/alarm_history_dump.parquet")
graph_repo = AlarmGraphRepository(os.getenv("HISTORY_DB_PATH"))

In [2]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationHistory

SimpleTimeCorrelationHistory.train(lazy_frame, graph_repo)

Processando nós: 100%|██████████| 912/912 [05:55<00:00,  2.56nó/s]  


In [2]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

Calculando WCC:  99%|█████████▉| 904/912 [02:40<00:01,  5.63nó/s] 


KeyboardInterrupt: 

In [ ]:
from src.utils.node_summary import node_summary
from src.repository.aggregate_results_repository import AggregateResultsRepository

summary, general_metrics = node_summary(graph_repo)

results_repo = AggregateResultsRepository(filename="simple_time_corr_history")
results_repo.save(summary)
results_repo.load()